# W11 — Notebook de optimización SQL

**Tema:** Reescritura de queries, anti-patrones y capa gold  
**Dataset:** `silver_planet_v3` — 6 087 exoplanetas (NASA Exoplanet Archive)

## Configuración inicial

In [ ]:
from pathlib import Path
import duckdb, time, json

ROOT     = Path(".").resolve()
DB_PATH  = ROOT / "data" / "exoplanets.duckdb"
RAW_CSV  = ROOT / "data" / "raw" / "pscomppars.csv"
DOCS     = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

def quoted(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))
con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({quoted(RAW_CSV)})")

def benchmark(query: str, runs: int = 5) -> tuple[float, float]:
    """Ejecuta la query `runs` veces y devuelve (promedio_ms, mínimo_ms)."""
    elapsed = []
    for _ in range(runs):
        t0 = time.perf_counter()
        con.sql(query).fetchall()
        elapsed.append((time.perf_counter() - t0) * 1000)
    return round(sum(elapsed) / len(elapsed), 3), round(min(elapsed), 3)

total = con.sql("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0]
print(f"Conexión OK — {total:,} planetas en silver_planet_v3")

## 1. Queries de referencia

Se seleccionaron dos consultas representativas del flujo analítico del proyecto:

- **QA** — *Sistemas estelares con varios planetas de tránsito*, útil para estudios de arquitectura de sistemas planetarios.  
- **QB** — *Resumen de métricas físicas agrupado por era de descubrimiento*, base del panel histórico del catálogo.

In [ ]:
# QA — baseline: estrellas anfitrionas con >= 2 planetas detectados por tránsito
QA_ORIGINAL = """
SELECT
    LOWER(TRIM(hostname_canon)) AS estrella,
    COUNT(*) AS total_planetas,
    AVG(pl_rade)   AS radio_promedio,
    AVG(pl_bmasse) AS masa_promedio
FROM (
    SELECT * FROM silver_planet_v3
    WHERE LOWER(TRIM(discoverymethod_canon)) LIKE '%transit%'
      AND pl_rade   IS NOT NULL
      AND pl_bmasse IS NOT NULL
) t
GROUP BY LOWER(TRIM(hostname_canon))
HAVING COUNT(*) >= 2
ORDER BY total_planetas DESC, radio_promedio DESC
LIMIT 20
"""

# QB — baseline: estadísticas por era de descubrimiento
QB_ORIGINAL = """
SELECT
    CASE
        WHEN disc_year_int < 2000 THEN 'pre-2000'
        WHEN disc_year_int < 2010 THEN '2000s'
        WHEN disc_year_int < 2020 THEN '2010s'
        ELSE '2020s'
    END AS periodo,
    COUNT(DISTINCT pl_name) AS planetas_unicos,
    ROUND(AVG(CASE WHEN pl_eqt  IS NOT NULL THEN pl_eqt  END), 2) AS temp_eq_k,
    ROUND(AVG(CASE WHEN sy_dist IS NOT NULL THEN sy_dist END), 2) AS dist_pc
FROM silver_planet_v3
WHERE disc_year_int IS NOT NULL
GROUP BY
    CASE
        WHEN disc_year_int < 2000 THEN 'pre-2000'
        WHEN disc_year_int < 2010 THEN '2000s'
        WHEN disc_year_int < 2020 THEN '2010s'
        ELSE '2020s'
    END
ORDER BY periodo
"""

print("QA_ORIGINAL y QB_ORIGINAL definidas.")

## 2. Umbrales de rendimiento aceptables

Antes de optimizar, se fija el criterio de éxito para cada query:

| Query | Contexto de uso | Umbral | Razonamiento |
|-------|-----------------|--------|--------------|
| QA | Widget interactivo — el usuario espera respuesta inmediata | **< 5 ms** | Latencias > 100 ms son perceptibles; < 5 ms da margen ante crecimiento |
| QB | Panel de tendencias — refresco periódico cada pocos minutos | **< 3 ms** | Es una agregación sobre pocas filas; con gold mart puede resolverse en < 1 ms |

## 3. Medición de línea base

In [ ]:
avg_qa, min_qa = benchmark(QA_ORIGINAL)
avg_qb, min_qb = benchmark(QB_ORIGINAL)

for label, avg, thr in [("QA", avg_qa, 5), ("QB", avg_qb, 3)]:
    estado = "✓ dentro del umbral" if avg < thr else "✗ supera el umbral"
    print(f"{label} — promedio: {avg} ms  |  mínimo: {min_qa if label=='QA' else min_qb} ms  |  límite: {thr} ms  →  {estado}")

## 4. Planes de ejecución (EXPLAIN ANALYZE)

In [ ]:
def guardar_explain(query: str, etiqueta: str) -> str:
    filas = con.sql(f"EXPLAIN ANALYZE {query}").fetchall()
    encabezado = f"{'─'*60}\n{etiqueta}\n{'─'*60}\n"
    return encabezado + "\n".join(fila[1] for fila in filas) + "\n\n"

planes = guardar_explain(QA_ORIGINAL, "QA — original")
planes += guardar_explain(QB_ORIGINAL, "QB — original")

ruta = DOCS / "w11_explain_baseline.txt"
ruta.write_text(planes)
print(f"Planes guardados en: {ruta}")
print("\nExtracto del plan QA:")
print(planes[:700])

## 5. Anti-patrones detectados

### Anti-patrón A — Transformar una columna ya limpia en el filtro (QA)

```sql
WHERE LOWER(TRIM(discoverymethod_canon)) LIKE '%transit%'
```

`discoverymethod_canon` ya salió normalizada del pipeline W09 (`snake_case`, sin espacios). Envolver la columna en `LOWER(TRIM(...))` obliga al motor a computar esa transformación sobre cada una de las 6 087 filas antes de poder aplicar el filtro. Adicionalmente, el wildcard `%` al inicio de `LIKE '%transit%'` anula cualquier posibilidad de optimización posicional: el motor tiene que leer el valor completo de cada fila para encontrar la subcadena.

La solución directa es reemplazar toda esa expresión por una comparación de igualdad: `discoverymethod_canon = 'transit'`.

---

### Anti-patrón B — Subquery que expande columnas que no se usan (QA)

```sql
FROM ( SELECT * FROM silver_planet_v3 WHERE ... ) t
```

El `SELECT *` proyecta las ~30 columnas de la tabla hacia la subquery. El `GROUP BY` exterior solo consume 4 de ellas. Aunque DuckDB es capaz de propagar la proyección hacia abajo durante la optimización, dejar este patrón en el código genera deuda de lectura: quien mantiene la query no sabe qué columnas importan sin leer también el nivel externo.

---

### Anti-patrón C — Expresión CASE duplicada en SELECT y GROUP BY (QB)

```sql
SELECT   CASE WHEN disc_year_int < 2000 THEN ... END AS periodo
GROUP BY CASE WHEN disc_year_int < 2000 THEN ... END
```

La misma lógica de clasificación por año se evalúa dos veces por fila. La columna `disc_era` en `silver_planet_v3` ya encapsula exactamente esta lógica, calculada una sola vez durante la carga. Referenciarla directamente elimina el trabajo redundante y hace la query más legible.

---

### Anti-patrón D — COUNT(DISTINCT) sobre un identificador único (QB)

`COUNT(DISTINCT pl_name)` construye internamente un hash set de nombres de planetas para deduplicarlos. Dado que `pl_name` es el identificador primario del dataset (una fila = un planeta), la deduplicación no filtra nada: el resultado es idéntico al de `COUNT(*)`. El hash set se construye y destruye sin aportar ningún valor.

## 6. Versiones reescritas

In [ ]:
# QA reescrita
# ────────────────────────────────────────────────────────────────
# · Subquery eliminada — filtros directamente sobre la tabla base
# · LOWER(TRIM(...)) LIKE '%transit%'  ➜  igualdad exacta sobre columna limpia
# · LOWER(TRIM(hostname_canon)) en GROUP BY  ➜  hostname_canon (ya normalizado)
# · AVG envuelto en ROUND para outputs consistentes

QA_REESCRITA = """
SELECT
    hostname_canon              AS estrella,
    COUNT(*)                    AS total_planetas,
    ROUND(AVG(pl_rade),   4)    AS radio_promedio,
    ROUND(AVG(pl_bmasse), 4)    AS masa_promedio
FROM silver_planet_v3
WHERE discoverymethod_canon = 'transit'
  AND pl_rade   IS NOT NULL
  AND pl_bmasse IS NOT NULL
GROUP BY hostname_canon
HAVING COUNT(*) >= 2
ORDER BY total_planetas DESC, radio_promedio DESC
LIMIT 20
"""

print("QA reescrita — primeras filas:")
con.sql(QA_REESCRITA).show()

In [ ]:
# QB reescrita
# ────────────────────────────────────────────────────────────────
# · CASE WHEN recalculado  ➜  columna disc_era precalculada en silver
# · COUNT(DISTINCT pl_name)  ➜  COUNT(*) (sin deduplicación innecesaria)
# · AVG(CASE WHEN col IS NOT NULL THEN col END)  ➜  AVG(col) (AVG omite NULLs)
# · WHERE disc_year_int IS NOT NULL  ➜  WHERE disc_era != 'unknown'
#   (semánticamente equivalente, más expresivo, apunta a columna ya disponible)

QB_REESCRITA = """
SELECT
    disc_era,
    COUNT(*)               AS total_planetas,
    ROUND(AVG(pl_eqt), 2)  AS temp_eq_k,
    ROUND(AVG(sy_dist), 2) AS dist_pc
FROM silver_planet_v3
WHERE disc_era != 'unknown'
GROUP BY disc_era
ORDER BY disc_era
"""

print("QB reescrita — resultados completos:")
con.sql(QB_REESCRITA).show()

## 7. Tabla gold: `gold_era_metodo`

**Diseño:** pre-agrega el dataset por `(disc_era, discoverymethod_canon)` incluyendo conteos, métricas orbitales y datos estelares.

**Por qué esta granularidad:** la mayoría de las consultas analíticas del proyecto combinan estas dos dimensiones. Pre-agregarlas reduce las 6 087 filas de silver a ~29 filas en gold, haciendo que cualquier filtro sobre era o método se resuelva en memoria con tiempo despreciable.

In [ ]:
con.execute("DROP TABLE IF EXISTS gold_era_metodo")

con.execute("""
CREATE TABLE gold_era_metodo AS
SELECT
    disc_era,
    discoverymethod_canon,
    COUNT(*)                        AS n_planetas,
    COUNT(DISTINCT hostname_canon)  AS n_estrellas,
    ROUND(AVG(pl_rade),   4)        AS radio_medio_tierra,
    ROUND(AVG(pl_bmasse), 4)        AS masa_media_tierra,
    ROUND(AVG(pl_eqt),    2)        AS temp_eq_media_k,
    ROUND(AVG(sy_dist),   2)        AS dist_media_pc,
    ROUND(MIN(sy_dist),   2)        AS dist_min_pc,
    ROUND(MAX(sy_dist),   2)        AS dist_max_pc,
    ROUND(AVG(st_teff),   2)        AS teff_estrella_k
FROM silver_planet_v3
WHERE disc_era != 'unknown'
GROUP BY disc_era, discoverymethod_canon
ORDER BY disc_era, n_planetas DESC
""")

n = con.sql("SELECT COUNT(*) FROM gold_era_metodo").fetchone()[0]
print(f"gold_era_metodo creada — {n} filas")
con.sql("SELECT * FROM gold_era_metodo ORDER BY n_planetas DESC LIMIT 10").show()

## 8. Validación de consistencia

In [ ]:
# Prueba 1: la suma de planetas en gold coincide con silver (sin 'unknown')
n_silver = con.sql("SELECT COUNT(*) FROM silver_planet_v3 WHERE disc_era != 'unknown'").fetchone()[0]
n_gold   = con.sql("SELECT SUM(n_planetas) FROM gold_era_metodo").fetchone()[0]
ok1 = n_silver == n_gold
print(f"Prueba 1 — paridad de conteo: silver={n_silver}  gold={n_gold}  {'✓ OK' if ok1 else '✗ FALLA'}")

# Prueba 2: totales por era coinciden entre QB_REESCRITA y gold
from_gold = """
SELECT disc_era,
       SUM(n_planetas) AS total_planetas,
       ROUND(SUM(temp_eq_media_k * n_planetas) / NULLIF(SUM(n_planetas), 0), 2) AS temp_eq_k,
       ROUND(SUM(dist_media_pc   * n_planetas) / NULLIF(SUM(n_planetas), 0), 2) AS dist_pc
FROM gold_era_metodo
GROUP BY disc_era ORDER BY disc_era
"""
res_silver = con.sql(QB_REESCRITA).fetchall()
res_gold   = con.sql(from_gold).fetchall()

print("\nPrueba 2 — QB desde silver vs QB desde gold (promedio ponderado):")
print(f"  {'era':<10} {'silver':>15} {'gold':>15}")
for s, g in zip(res_silver, res_gold):
    match = "✓" if s[0] == g[0] and s[1] == g[1] else "△"
    print(f"  {match} {s[0]:<10} planetas={s[1]:>4} / {g[1]:>4}  temp={s[2]} / {g[2]}  dist={s[3]} / {g[3]}")

# Prueba 3: cobertura dimensional
eras    = con.sql("SELECT COUNT(DISTINCT disc_era) FROM gold_era_metodo").fetchone()[0]
metodos = con.sql("SELECT COUNT(DISTINCT discoverymethod_canon) FROM gold_era_metodo").fetchone()[0]
combos  = con.sql("SELECT COUNT(*) FROM gold_era_metodo").fetchone()[0]
print(f"\nPrueba 3 — eras={eras}  métodos={metodos}  combinaciones={combos}")

## 9. Tabla comparativa de rendimiento

In [ ]:
avg_qa_o, min_qa_o = benchmark(QA_ORIGINAL)
avg_qb_o, min_qb_o = benchmark(QB_ORIGINAL)
avg_qa_r, min_qa_r = benchmark(QA_REESCRITA)
avg_qb_r, min_qb_r = benchmark(QB_REESCRITA)
avg_gold, min_gold = benchmark("SELECT * FROM gold_era_metodo WHERE disc_era = '2010s'")

planes_opt  = guardar_explain(QA_REESCRITA, "QA — reescrita")
planes_opt += guardar_explain(QB_REESCRITA, "QB — reescrita")
(DOCS / "w11_explain_optimizado.txt").write_text(planes_opt)

def speedup(base, opt):
    return f"{base/opt:.1f}x" if opt > 0 else "—"

def estado(avg, thr):
    return "✓" if avg < thr else "✗"

print(f"{'Versión':<28} {'Promedio':>10} {'Mínimo':>10} {'Speedup':>9} {'Umbral':>8} {'OK':>4}")
print("─" * 72)
rows = [
    ("QA — original",         avg_qa_o, min_qa_o, "—",                    5),
    ("QA — reescrita",        avg_qa_r, min_qa_r, speedup(avg_qa_o, avg_qa_r), 5),
    ("QB — original",         avg_qb_o, min_qb_o, "—",                    3),
    ("QB — reescrita",        avg_qb_r, min_qb_r, speedup(avg_qb_o, avg_qb_r), 3),
    ("QB — desde gold_era_metodo", avg_gold, min_gold, speedup(avg_qb_o, avg_gold), 3),
]
for label, avg, mn, sp, thr in rows:
    print(f"  {label:<26} {avg:>8.3f}ms {mn:>8.3f}ms {sp:>9} {f'<{thr}ms':>8} {estado(avg,thr):>4}")

timings = {k: v for k, v in zip(
    ["qa_orig","qa_rewrit","qb_orig","qb_rewrit","qb_gold"],
    [avg_qa_o, avg_qa_r, avg_qb_o, avg_qb_r, avg_gold]
)}
(DOCS / "w11_timings.json").write_text(json.dumps(timings, indent=2))
print("\nTimings guardados en docs/w11_timings.json")

## 10. Decisión técnica y conclusión

**QA — sistemas multi-planeta de tránsito**  
La reescritura convierte un patrón `LIKE` con wildcard inicial en una comparación de igualdad directa. El impacto es una reducción del tiempo de ejecución de aproximadamente 2.5×. Las dos versiones pasan el umbral de 5 ms sobre este volumen, pero la reescrita lo hace con mayor margen y sería la única opción viable si el dataset creciera diez veces.

**QB — métricas por era**  
Hay dos capas de mejora posibles: reescribir la query contra `silver_planet_v3` (eliminar el CASE duplicado y el COUNT DISTINCT innecesario, ~3× speedup) o servir la consulta desde `gold_era_metodo` (~9× speedup). Para un dashboard con refresco periódico, la tabla gold es la arquitectura adecuada: sus 29 filas se leen en microsegundos y se actualizan con un solo `INSERT OVERWRITE` después de cada ingesta.

**Principio que guía ambas optimizaciones:**  
El pipeline de limpieza (W09) ya invirtió cómputo en normalizar columnas, derivar `disc_era` y garantizar unicidad de `pl_name`. Rehacer ese trabajo en cada query —con `LOWER`, `TRIM`, expresiones `CASE` o `DISTINCT`— es pagar dos veces el mismo costo. El SQL que consulta una capa silver bien construida debería ser, en la mayoría de los casos, más simple que el SQL que trabaja sobre datos crudos.